# Step 6b — Tail calibration: why Policy E did not win

`docs/RESULTS.md` section 3 leaves an open question. Policy E (`alert iff
p * Amount > c_review`) is the Bayes-optimal rule for the project's cost
objective, yet on test it did not beat a tuned global threshold (Policy A):

| Model | Policy A | Policy E | E - A (paired bootstrap) |
|---|---|---|---|
| xgb/balanced | EUR2,223.93 | EUR2,316.64 | +EUR100.95, not significant |
| rf/none | EUR2,453.93 | EUR2,415.60 | -EUR16.52, not significant |
| logreg/none | EUR2,231.31 | EUR2,431.08 | +EUR199.23, **significant, E worse** |

Aggregate calibration is good for all three (mean predicted `p` matches the
0.172% base rate within 0.7x-1.0x), so simple miscalibration does not explain
this. Policy E alert counts also differ wildly on test (56,962 rows): xgb 46,
rf 396, logreg 166.

**Hypothesis under test.** Policy E compares `p` against the
*per-transaction* threshold `c_review/Amount`: EUR0.3 for a EUR10 transaction,
EUR0.003 for a EUR1,000 one. Aggregate calibration only checks that the mean
of `p` matches the mean base rate -- it says nothing about whether `p` is
accurate at these specific, mostly very low, thresholds. A model can look
perfectly calibrated on average while being systematically wrong exactly
where Policy E reads it.

This notebook does not assume the hypothesis is true. It is tested against
the cached test/validation probability arrays, with a plain verdict at the
end.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

import fraud_cost
from fraud_cost import (policy_e_predict, total_cost, optimal_threshold,
                        undo_class_weight)

sns.set_theme(style="whitegrid")
C_REVIEW = 3.0

# Anchor to the project root via the importable module, so this works
# whether the kernel cwd is /work or /work/notebooks.
ROOT = Path(fraud_cost.__file__).resolve().parent
ART = ROOT / "artifacts"

te = np.load(ART / "test_probabilities.npz")
va = np.load(ART / "val_probabilities.npz")
yte, amt_te = te["y"], te["amounts"]
yva, amt_va = va["y"], va["amounts"]
prereg = json.loads((ART / "preregistration.json").read_text())

MODELS = ["xgb/balanced", "rf/none", "logreg/none"]
print(f"test set : {len(yte):,} rows, {int(yte.sum())} frauds, "
      f"fraud value EUR{amt_te[yte == 1].sum():,.2f}")
print(f"val set  : {len(yva):,} rows, {int(yva.sum())} frauds")
print(f"pre-registered champion: {prereg['declared_winner']}")


## Setup — recover the posterior `p` that Policy E actually consumes

`xgb/balanced` was trained with `class_weight`-style rebalancing, which
inflates predicted probabilities by roughly the inverse class ratio (see
`fraud_cost.undo_class_weight`). Policy E, and any calibration check *of*
Policy E, must use the corrected posterior -- not the raw reweighted score.
`rf/none` and `logreg/none` need no correction. This mirrors exactly what
`analyse_results.py` does before calling `policy_e_predict`.

In [ ]:
def posterior(split_arr, y_arr, key):
    """Return the probability actually fed to Policy E for `key`."""
    p = split_arr[key]
    if key.endswith("/balanced"):
        w = (y_arr == 0).sum() / (y_arr == 1).sum()
        return undo_class_weight(p, w)
    return p


P_TEST = {k: posterior(te, yte, k) for k in MODELS}
P_VAL = {k: posterior(va, yva, k) for k in MODELS}

# Sanity check: reproduce the Policy E test costs from RESULTS.md section 3
# before doing anything new.
print("Policy E cost, recomputed from cached arrays (must match RESULTS.md):")
for k in MODELS:
    pred = policy_e_predict(P_TEST[k], amt_te, C_REVIEW)
    cost = total_cost(yte, pred, amt_te, C_REVIEW)
    print(f"  {k:<14} EUR{cost:>10,.2f}   alerts {int(pred.sum()):>4}  "
          f"({pred.mean()*100:.3f}%)")


## 1. Reliability curves, log-spaced in the low-probability region

At a 0.172% base rate almost all probability mass sits below `p = 0.01`.
Uniform-width bins (the default for `sklearn.calibration.calibration_curve`)
would put nearly every transaction in the single `[0, 0.1)` bin and say
nothing about the tail. Log-spaced bins spread that mass out instead. Exact
zeros are pooled into their own bin -- `rf` emits only ~101 distinct
probability values (a tree-vote fraction), so most of its mass is an exact
`0.0`, not a small positive number.

In [ ]:
def reliability_log(p, y, edges):
    """Calibration table over log-spaced bins, with an explicit p==0 bin.

    edges[0] must be 0. Returns one row per non-empty bin: predicted mean,
    observed fraud rate, count, and the calibration ratio (pred / observed).
    """
    assert edges[0] == 0
    bin_idx = np.searchsorted(edges, p, side="right") - 1
    bin_idx = np.clip(bin_idx, 0, len(edges) - 2)
    rows = []
    for b in range(len(edges) - 1):
        mask = bin_idx == b
        n = int(mask.sum())
        if n == 0:
            continue
        obs = y[mask].mean()
        pred = p[mask].mean()
        rows.append({
            "bin_lo": edges[b], "bin_hi": edges[b + 1], "n": n,
            "n_fraud": int(y[mask].sum()),
            "pred_mean": pred, "observed_rate": obs,
            "ratio": (pred / obs) if obs > 0 else np.nan,
        })
    return pd.DataFrame(rows)


EDGES = np.concatenate([[0.0], np.logspace(-6, 0, 19)])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=False)
tables = {}
for ax, k in zip(axes, MODELS):
    tab = reliability_log(P_TEST[k], yte, EDGES)
    tables[k] = tab
    mid = np.sqrt(np.maximum(tab.bin_lo, 1e-7) * tab.bin_hi)
    ax.plot([1e-7, 1], [1e-7, 1], "k--", lw=1, label="perfect calibration")
    sc = ax.scatter(tab.pred_mean.clip(lower=1e-7), tab.observed_rate.clip(lower=1e-7),
                     s=np.clip(tab.n, 5, None) ** 0.5 * 4, c=tab.n_fraud,
                     cmap="Reds", edgecolor="black", linewidth=.4, zorder=3)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(1e-7, 1.5); ax.set_ylim(1e-7, 1.5)
    ax.set_xlabel("mean predicted p (bin)")
    ax.set_title(k)
    ax.axvline(3 / amt_te[amt_te > 0].max(), color="grey", ls=":", lw=1)
    ax.axvline(min(1.0, 3 / np.percentile(amt_te[amt_te > 0], 1)), color="grey", ls=":", lw=1)
axes[0].set_ylabel("observed fraud rate (bin)")
axes[0].legend(loc="upper left", fontsize=8)
fig.suptitle("Reliability curves, log-spaced bins (dot size ~ n, colour ~ fraud count in bin)\n"
             "dotted grey lines mark the c_review/Amount range Policy E actually reads")
plt.tight_layout(); plt.show()

for k in MODELS:
    print(f"\n=== {k} ===")
    display(tables[k].round(6))


**Reading the plot.** The grey dotted lines on each panel mark the
`c_review/Amount` range Policy E evaluates against (from the largest test
Amount up to the 99th-percentile Amount). Aggregate calibration collapses all
bins into one weighted-average ratio; here each bin is shown separately, so a
systematic tilt in exactly that region is visible even if it cancels out on
average.

## 2. Calibration exactly at the thresholds Policy E uses

Section 1 bins by predicted probability. This section bins by the
**per-transaction threshold** `t_i = c_review / Amount_i` instead -- i.e. by
Amount, since larger Amount means a lower, stricter threshold. Within each
Amount decile we get a calibration ratio (`mean predicted p` / `observed
fraud rate`) exactly analogous to the aggregate 0.7x/1.0x/1.0x ratios in
`RESULTS.md`, but computed locally. `Amount == 0` rows have `t_i = +inf` --
Policy E can never alert on them regardless of `p`, so they are excluded
here (5 such frauds on test, matching the PLAN's zero-Amount note).

In [ ]:
nonzero = amt_te > 0
t_i = np.full(len(amt_te), np.inf)
t_i[nonzero] = C_REVIEW / amt_te[nonzero]

deciles = pd.qcut(amt_te[nonzero], 10, duplicates="drop")

rows = []
for k in MODELS:
    p = P_TEST[k][nonzero]
    y = yte[nonzero]
    a = amt_te[nonzero]
    t = t_i[nonzero]
    df = pd.DataFrame({"amount": a, "t": t, "p": p, "y": y, "decile": deciles})
    for dec, g in df.groupby("decile", observed=True):
        obs = g.y.mean()
        pred = g.p.mean()
        rows.append({
            "model": k,
            "amount_range": f"[{g.amount.min():.2f}, {g.amount.max():.2f}]",
            "median_threshold_t=c/Amount": g.t.median(),
            "n": len(g), "n_fraud": int(g.y.sum()),
            "mean_pred_p": pred, "observed_rate": obs,
            "ratio_pred/observed": (pred / obs) if obs > 0 else np.nan,
            "policy_e_alerts": int((g.p > g.t).sum()),
        })

by_amount = pd.DataFrame(rows)
for k in MODELS:
    print(f"\n=== {k} : calibration by Amount decile (= by Policy E threshold) ===")
    display(by_amount[by_amount.model == k].drop(columns="model").round(6)
            .sort_values("median_threshold_t=c/Amount", ascending=False))


**Boundary-adjacent calibration.** The decile table mixes transactions whose
`p` is nowhere near their own threshold (an easy legit transaction, or an
obvious fraud) with the ones whose decision Policy E actually agonises over.
Restricting to transactions where `p` sits within a factor of 3 of their own
`t_i` isolates the second group -- these are the rows a small calibration
error can flip from alert to no-alert or back.

In [ ]:
print("Calibration ratio restricted to the decision boundary (0.33x <= p/t_i <= 3x):\n")
for k in MODELS:
    p = P_TEST[k][nonzero]
    y = yte[nonzero]
    t = t_i[nonzero]
    ratio = p / t
    near = (ratio >= 1/3) & (ratio <= 3)
    n = int(near.sum())
    if n == 0:
        print(f"  {k:<14} no transactions within 3x of their own threshold")
        continue
    obs = y[near].mean()
    pred = p[near].mean()
    print(f"  {k:<14} n={n:>4}  n_fraud={int(y[near].sum()):>3}  "
          f"mean_p={pred:.5f}  observed_rate={obs:.5f}  "
          f"ratio={pred/obs if obs>0 else float('nan'):.2f}x  "
          f"(cf. aggregate ratio {P_TEST[k].mean()/yte.mean():.2f}x)")


## 3. Decomposing Policy E's cost: FP review cost vs FN lost Amount

`rf/none` fires 396 alerts on test. Is `396 * EUR3 = EUR1,188` of review cost
the whole story, or is something else driving its Policy E cost? Same
question for the other two models. `TotalCost = c_review*(TP+FP) +
sum(Amount over FN)`, so this splits cleanly into three pieces.

In [ ]:
rows = []
for k in MODELS:
    pred = policy_e_predict(P_TEST[k], amt_te, C_REVIEW)
    tp = int(((pred == 1) & (yte == 1)).sum())
    fp = int(((pred == 1) & (yte == 0)).sum())
    fn_mask = (pred == 0) & (yte == 1)
    fn = int(fn_mask.sum())
    fn_amount = float(amt_te[fn_mask].sum())
    tp_cost = C_REVIEW * tp
    fp_cost = C_REVIEW * fp
    total = tp_cost + fp_cost + fn_amount
    rows.append({
        "model": k, "alerts": int(pred.sum()), "TP": tp, "FP": fp, "FN": fn,
        "TP_review_cost": tp_cost, "FP_review_cost": fp_cost,
        "FN_lost_amount": fn_amount, "total_(check)": total,
    })

decomp = pd.DataFrame(rows).set_index("model")
display(decomp.round(2))

print()
for k in MODELS:
    r = decomp.loc[k]
    print(f"{k:<14} review cost (TP+FP) = EUR{r.TP_review_cost + r.FP_review_cost:>9,.2f}   "
          f"missed-fraud cost (FN) = EUR{r.FN_lost_amount:>9,.2f}")


## 4. Does recalibration fix Policy E?

Fit a monotone recalibrator on **validation** `(p, y)` only, then re-evaluate
Policy E on test with the recalibrated probabilities. Two recalibrators:

- **Isotonic regression** -- flexible, can correct a region-specific tilt
  that a global rescaling cannot, at the cost of collapsing tied inputs onto
  plateaus.
- **Platt scaling** (a logistic regression of `y` on `logit(p)`) -- a smooth
  monotone rescaling; per `docs/PLAN.md` this is a no-op for Policy A (which
  re-derives its threshold after any monotone transform), but Policy E
  compares against a *fixed* external quantity (`c_review/Amount`), so even a
  monotone rescaling can change its decisions.

Both are fit on `P_VAL` (the same posterior-corrected probabilities Policy E
already consumes), never on test.

In [ ]:
def platt_transform(p_fit, y_fit, p_apply):
    eps = 1e-9
    logit = np.log(np.clip(p_fit, eps, 1 - eps) / np.clip(1 - p_fit, eps, 1 - eps))
    lr = LogisticRegression()
    lr.fit(logit.reshape(-1, 1), y_fit)
    logit_apply = np.log(np.clip(p_apply, eps, 1 - eps) / np.clip(1 - p_apply, eps, 1 - eps))
    return lr.predict_proba(logit_apply.reshape(-1, 1))[:, 1]


recal_rows = []
for k in MODELS:
    p_val, y_val = P_VAL[k], yva
    p_test = P_TEST[k]

    iso = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip")
    iso.fit(p_val, y_val)
    p_test_iso = iso.predict(p_test)

    p_test_platt = platt_transform(p_val, y_val, p_test)

    baseline_pred = policy_e_predict(p_test, amt_te, C_REVIEW)
    iso_pred = policy_e_predict(p_test_iso, amt_te, C_REVIEW)
    platt_pred = policy_e_predict(p_test_platt, amt_te, C_REVIEW)

    recal_rows.append({
        "model": k,
        "E_cost_baseline": total_cost(yte, baseline_pred, amt_te, C_REVIEW),
        "E_alerts_baseline": int(baseline_pred.sum()),
        "E_cost_isotonic": total_cost(yte, iso_pred, amt_te, C_REVIEW),
        "E_alerts_isotonic": int(iso_pred.sum()),
        "E_cost_platt": total_cost(yte, platt_pred, amt_te, C_REVIEW),
        "E_alerts_platt": int(platt_pred.sum()),
    })

recal = pd.DataFrame(recal_rows).set_index("model")
display(recal.round(2))

policy_a_cost = {"xgb/balanced": 2223.93, "rf/none": 2453.93, "logreg/none": 2231.31}
print("\nRecalibrated Policy E vs Policy A (test, from RESULTS.md):")
for k in MODELS:
    r = recal.loc[k]
    best_e = min(r.E_cost_isotonic, r.E_cost_platt)
    print(f"  {k:<14} Policy A EUR{policy_a_cost[k]:>9,.2f}   "
          f"E baseline EUR{r.E_cost_baseline:>9,.2f}   "
          f"E best-recalibrated EUR{best_e:>9,.2f}   "
          f"({'closes gap' if best_e < r.E_cost_baseline else 'no improvement'})")


Sanity check on the recalibration itself: does isotonic regression actually
change the reliability curve in the region that matters, or does it barely
move `p` at all (in which case a null result above is unsurprising rather
than informative)?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, k in zip(axes, MODELS):
    iso = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip")
    iso.fit(P_VAL[k], yva)
    p_test_iso = iso.predict(P_TEST[k])

    before = reliability_log(P_TEST[k], yte, EDGES)
    after = reliability_log(p_test_iso, yte, EDGES)

    ax.plot([1e-7, 1], [1e-7, 1], "k--", lw=1)
    ax.scatter(before.pred_mean.clip(lower=1e-7), before.observed_rate.clip(lower=1e-7),
               s=25, color="#4C78A8", label="before (baseline p)", alpha=.8)
    ax.scatter(after.pred_mean.clip(lower=1e-7), after.observed_rate.clip(lower=1e-7),
               s=25, color="#E45756", label="after (isotonic)", alpha=.8)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(1e-7, 1.5); ax.set_ylim(1e-7, 1.5)
    ax.set_title(k); ax.set_xlabel("mean predicted p (bin)")
axes[0].set_ylabel("observed fraud rate (bin)")
axes[0].legend(fontsize=8)
fig.suptitle("Effect of isotonic recalibration (fit on validation, applied to test) on reliability")
plt.tight_layout(); plt.show()


## 5. Verdict

This section states the conclusion in the same terms `RESULTS.md` section 3
poses the question, using the numbers this notebook produced (re-run the
cells above to reproduce them).

In [ ]:
print("=== VERDICT INPUTS (read off the tables above before writing the verdict) ===\n")

print("Aggregate calibration ratio (mean p / base rate), test set:")
for k in MODELS:
    print(f"  {k:<14} {P_TEST[k].mean() / yte.mean():.3f}x")

print("\nCalibration ratio near each model's own decision boundary "
      "(0.33x-3x of t_i), from section 2:")
for k in MODELS:
    p = P_TEST[k][nonzero]; y = yte[nonzero]; t = t_i[nonzero]
    ratio = p / t
    near = (ratio >= 1/3) & (ratio <= 3)
    obs = y[near].mean() if near.sum() else float("nan")
    pred = p[near].mean() if near.sum() else float("nan")
    print(f"  {k:<14} n_near={int(near.sum()):>4}  "
          f"ratio={pred/obs if obs else float('nan'):.2f}x")

print("\nPolicy E cost: baseline vs best recalibrated vs Policy A:")
for k in MODELS:
    r = recal.loc[k]
    best_e = min(r.E_cost_isotonic, r.E_cost_platt)
    print(f"  {k:<14} A={policy_a_cost[k]:>9,.2f}  "
          f"E_base={r.E_cost_baseline:>9,.2f}  E_recal={best_e:>9,.2f}")


### Written verdict

**Partly supported.** Tail/region calibration errors are real, they are
invisible to the aggregate calibration ratio, and their *sign* at each
model's own decision boundary predicts the *sign* of that model's Policy E
vs Policy A gap from `RESULTS.md`. But a standard fix -- post-hoc
recalibration -- does not reliably repair it, for a reason that also
explains why it exists in the first place: there is not enough tail data at
this fraud rate.

**1. Aggregate calibration is confirmed good, exactly as `RESULTS.md` reports**
(ratios 0.71x / 1.04x / 1.02x for xgb / rf / logreg). That is real, and it is
also the wrong place to look.

**2. Per-Amount-decile calibration (Section 2) swings far from 1x while the
aggregate stays near 1x.** logreg ranges 0.23x-2.47x across deciles despite an
aggregate of 1.02x; rf ranges 0.68x-2.94x despite 1.04x; xgb is consistently
*underconfident* in every single decile (0.39x-0.94x, never above 1x) which is
also why its aggregate (0.71x) is already off. "Good on average" and "good in
the region Policy E reads" are different claims, and the difference is large.

**3. Calibration restricted to each model's own decision boundary** (p within
3x of its transaction's `c_review/Amount`, Section 2) is 0.77x (xgb), 1.46x
(rf), 0.70x (logreg) -- and this number lines up with each model's Policy E
cost decomposition (Section 3):
- **logreg** is *underconfident* at its boundary (0.70x) -> fraud that should
  score just above threshold scores just below it and is missed. EUR1,933 of
  its EUR2,431 Policy E cost (79%) is missed-fraud (FN), not review cost.
  logreg is the model where `RESULTS.md` reports E **significantly** worse
  than A -- the one case with a real effect to explain, and this is a
  plausible mechanism for it.
- **xgb** is underconfident everywhere, boundary included (0.77x, on top of
  an already-low 0.71x aggregate) -> extremely conservative (46 alerts on
  56,962 rows), EUR2,179 of its EUR2,317 Policy E cost (94%) is missed fraud.
  Consistent with the (not-significant) E > A gap `RESULTS.md` reports.
- **rf** is *overconfident* at its boundary (1.46x) -> more alerts than
  warranted, concentrated in the top Amount decile (327 of its 396 alerts,
  83%, fire there alone). Its cost splits nearly evenly between review
  (EUR1,188) and missed fraud (EUR1,228) -- rf is also the one model where
  Policy E beat Policy A on test (EUR2,415.60 vs EUR2,453.93), and it is also
  the one model whose boundary calibration is overconfident rather than
  underconfident.

**4. Recalibration does not fix this** (Section 4). Isotonic/Platt scaling
fit on the ~99 validation frauds *increased* Policy E's test cost for xgb
(EUR2,316.64 -> EUR2,462.70) and logreg (EUR2,431.08 -> EUR2,501.09), and only
partially helped rf (EUR2,415.60 -> EUR2,360.70). Two out of three
recalibration attempts made things worse, not better.

**Reading the two findings together:** this is not "the models are badly
calibrated, fix the calibration." With 98 test frauds and 99 validation
frauds spread across ten Amount deciles (many deciles hold single digits of
fraud among ~5,600 transactions), *any* calibrator -- the original model or a
post-hoc isotonic/Platt fit -- is estimating a low-probability tail from a
handful of positive examples per region. That is also precisely why the A-vs-E
gaps were mostly statistically insignificant to begin with (n_eff ~= 18,
`RESULTS.md` section 2). The tail-calibration hypothesis correctly identifies
*where* the models diverge from the Bayes-optimal assumption Policy E needs;
it does not identify a *fixable bug*, because the underlying constraint is
data volume, not a correctable systematic error. The report should present
this as: Policy E's theoretical optimality is undermined by fine-grained
calibration error that aggregate metrics cannot see and that recalibration,
given so few positive examples, cannot reliably correct either.